# Evaluación y comparación de los sistemas de recomendación

En este notebook se realiza la evaluación final de los diferentes sistemas de recomendación desarrollados.

El objetivo es analizar tanto su capacidad para predecir las valoraciones de los usuarios como la calidad de las recomendaciones generadas.

Para ello se utilizarán métricas de error de predicción y métricas orientadas a la generación de recomendaciones Top-N.

## 1. Preparación del entorno

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow import keras
import pickle

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 2. Carga de las particiones de datos

Para realizar la evaluación se utilizan las particiones de entrenamiento, validación y test generadas durante el desarrollo del modelo de Deep Learning.

La reutilización de estas particiones permite mantener una división reproducible de los datos y establecer un conjunto de test común sobre el que comparar los diferentes modelos.

In [3]:
ruta_modelos = "/content/drive/MyDrive/TFM/modelos"

train = pd.read_csv(f"{ruta_modelos}/train.csv")
validation = pd.read_csv(f"{ruta_modelos}/validation.csv")
test = pd.read_csv(f"{ruta_modelos}/test.csv")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (80668, 6)
Validation: (10084, 6)
Test: (10084, 6)


### 2.1. Creación del conjunto de test común

Los modelos colaborativos necesitan disponer de información previa sobre los usuarios y películas para poder realizar predicciones fundamentadas.

Por este motivo, se crea un conjunto de test común formado únicamente por interacciones cuyos usuarios y películas aparecen en el conjunto de entrenamiento.

Este conjunto será utilizado para comparar los modelos bajo las mismas condiciones.

In [4]:
usuarios_train = set(train["userId"])
peliculas_train = set(train["movieId"])

test_comun = test[
    test["userId"].isin(usuarios_train)
    & test["movieId"].isin(peliculas_train)
].copy()

print("Test original:", len(test))
print("Test común evaluable:", len(test_comun))
print(
    "Porcentaje conservado:",
    round(len(test_comun) / len(test) * 100, 2),
    "%"
)

Test original: 10084
Test común evaluable: 9695
Porcentaje conservado: 96.14 %


## 3. Evaluación del modelo Deep Learning híbrido

Se comienza evaluando el modelo híbrido seleccionado durante la fase de desarrollo.

Para ello se carga el modelo almacenado y se reproduce sobre el conjunto de test común el mismo preprocesamiento utilizado durante el entrenamiento.

### 3.1. Carga del modelo y del preprocesamiento

El modelo necesita recuperar tanto los pesos aprendidos como los mapeos utilizados para transformar los identificadores originales de usuarios y películas en sus correspondientes índices internos.

In [6]:
modelo_dl = keras.models.load_model(
    f"{ruta_modelos}/modelo_deep_learning.keras"
)

In [7]:
with open(
    f"{ruta_modelos}/usuario_a_indice.pkl",
    "rb"
) as f:
    usuario_a_indice = pickle.load(f)

with open(
    f"{ruta_modelos}/pelicula_a_indice.pkl",
    "rb"
) as f:
    pelicula_a_indice = pickle.load(f)

with open(
    f"{ruta_modelos}/columnas_generos.pkl",
    "rb"
) as f:
    columnas_generos = pickle.load(f)

In [8]:
preferencias_usuario = pd.read_csv(
    f"{ruta_modelos}/preferencias_usuario.csv"
)

In [9]:
print("Modelo cargado correctamente")
print("Usuarios mapeados:", len(usuario_a_indice))
print("Películas mapeadas:", len(pelicula_a_indice))
print("Géneros:", len(columnas_generos))
print("Perfiles de usuario:", preferencias_usuario.shape)

Modelo cargado correctamente
Usuarios mapeados: 610
Películas mapeadas: 8983
Géneros: 20
Perfiles de usuario: (610, 21)


### 3.2. Preparación del conjunto de test

Para evaluar el modelo se reproducen sobre el conjunto de test común las mismas transformaciones utilizadas durante el entrenamiento.

Cada interacción se representa mediante cuatro entradas: el índice interno del usuario, el índice interno de la película, los géneros de la película y el perfil de preferencias de género del usuario.

In [10]:
test_dl = test_comun.copy()

test_dl["usuario_idx"] = (
    test_dl["userId"].map(usuario_a_indice)
)

test_dl["pelicula_idx"] = (
    test_dl["movieId"].map(pelicula_a_indice)
)

print(
    test_dl[["usuario_idx", "pelicula_idx"]]
    .isna()
    .sum()
)

usuario_idx     0
pelicula_idx    0
dtype: int64


In [12]:
!find /content/drive/MyDrive -name "movies.csv"

/content/drive/MyDrive/TFM/Recomendador_Peliculas/data/raw/ml-latest-small/movies.csv


In [13]:
movies = pd.read_csv(
    "/content/drive/MyDrive/TFM/Recomendador_Peliculas/data/raw/ml-latest-small/movies.csv"
)

print(movies.shape)
display(movies.head())

(9742, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [14]:
generos_test = movies["genres"].str.get_dummies(sep="|")

movies_generos_test = pd.concat(
    [
        movies[["movieId"]],
        generos_test
    ],
    axis=1
)

test_dl = test_dl.merge(
    movies_generos_test,
    on="movieId",
    how="left"
)

In [15]:
test_dl = test_dl.merge(
    preferencias_usuario,
    on="userId",
    how="left"
)

In [16]:
columnas_preferencias = [
    f"pref_{genero}"
    for genero in columnas_generos
]

X_test_usuario = (
    test_dl["usuario_idx"]
    .to_numpy()
    .astype("int32")
)

X_test_pelicula = (
    test_dl["pelicula_idx"]
    .to_numpy()
    .astype("int32")
)

X_test_generos = (
    test_dl[columnas_generos]
    .to_numpy()
    .astype("float32")
)

X_test_preferencias = (
    test_dl[columnas_preferencias]
    .to_numpy()
    .astype("float32")
)

y_test = (
    test_dl["rating"]
    .to_numpy()
    .astype("float32")
)

In [17]:
print("Usuarios:", X_test_usuario.shape)
print("Películas:", X_test_pelicula.shape)
print("Géneros:", X_test_generos.shape)
print("Preferencias:", X_test_preferencias.shape)
print("Ratings:", y_test.shape)

print("NaN géneros:", np.isnan(X_test_generos).sum())
print("NaN preferencias:", np.isnan(X_test_preferencias).sum())

Usuarios: (9695,)
Películas: (9695,)
Géneros: (9695, 20)
Preferencias: (9695, 20)
Ratings: (9695,)
NaN géneros: 0
NaN preferencias: 0


### 3.3. Predicción y métricas de error

Una vez preparado el conjunto de test, el modelo seleccionado genera una predicción para cada interacción.

El rendimiento se evalúa mediante MAE (Mean Absolute Error) y RMSE (Root Mean Squared Error). El MAE representa el error absoluto medio de las predicciones, mientras que el RMSE penaliza con mayor intensidad los errores de mayor magnitud.

Ambas métricas se expresan en la misma escala que las valoraciones originales, por lo que valores menores indican una mayor precisión.

In [18]:
predicciones_dl = modelo_dl.predict(
    [
        X_test_usuario,
        X_test_pelicula,
        X_test_generos,
        X_test_preferencias
    ],
    verbose=0
).flatten()

print("Número de predicciones:", len(predicciones_dl))
print("Primeras predicciones:", predicciones_dl[:10])

Número de predicciones: 9695
Primeras predicciones: [4.1235876 4.417764  4.405495  4.378837  4.0852537 3.5840201 3.8156843
 3.7097344 3.2086513 3.3599708]


In [19]:
mae_dl = mean_absolute_error(
    y_test,
    predicciones_dl
)

rmse_dl = np.sqrt(
    mean_squared_error(
        y_test,
        predicciones_dl
    )
)

print("MAE Deep Learning:", round(mae_dl, 4))
print("RMSE Deep Learning:", round(rmse_dl, 4))

MAE Deep Learning: 0.6462
RMSE Deep Learning: 0.8527


In [20]:
print(
    "Predicción mínima:",
    round(predicciones_dl.min(), 3)
)

print(
    "Predicción máxima:",
    round(predicciones_dl.max(), 3)
)

Predicción mínima: 0.353
Predicción máxima: 5.45


#### Resultados de predicción

El modelo Deep Learning híbrido obtiene sobre el conjunto de test común un MAE de 0.6462 y un RMSE de 0.8527.

El MAE indica que las predicciones se desvían de las valoraciones reales aproximadamente 0.65 puntos de media en una escala comprendida entre 0.5 y 5.

Por su parte, el RMSE alcanza un valor de 0.8527. La proximidad de este resultado al obtenido durante la fase de validación indica que el comportamiento del modelo se mantiene sobre datos no utilizados durante el desarrollo y selección de la arquitectura.

La salida lineal del modelo puede producir ocasionalmente valores ligeramente fuera del intervalo de valoraciones permitido. En el conjunto de test se obtuvo un mínimo de 0.353 y un máximo de 5.45. Para la generación de recomendaciones finales, estas predicciones pueden limitarse al intervalo [0.5, 5].

## 4. Evaluación del modelo de factores latentes

El modelo basado en factores latentes se evalúa sobre el mismo conjunto de test común utilizado para el modelo Deep Learning.

De esta forma, ambos modelos realizan predicciones sobre exactamente las mismas interacciones usuario-película, permitiendo comparar directamente sus métricas de error.

In [21]:
usuarios = train["userId"].unique()
peliculas = train["movieId"].unique()

user_to_idx = {
    usuario: idx
    for idx, usuario in enumerate(usuarios)
}

movie_to_idx = {
    pelicula: idx
    for idx, pelicula in enumerate(peliculas)
}

n_usuarios = len(usuarios)
n_peliculas = len(peliculas)

print("Usuarios:", n_usuarios)
print("Películas:", n_peliculas)

Usuarios: 610
Películas: 8983


### 4.1. Preparación de los datos

Para garantizar una comparación homogénea, el modelo de factores latentes se entrena utilizando el mismo conjunto de entrenamiento empleado en la evaluación del modelo Deep Learning.

Se mantiene la configuración seleccionada durante el desarrollo previo del modelo: 20 factores latentes, una tasa de aprendizaje de 0.01, regularización de 0.05 y 21 épocas de entrenamiento.

In [22]:
train_mf = train.copy()

train_mf["user_idx"] = (
    train_mf["userId"].map(user_to_idx)
)

train_mf["movie_idx"] = (
    train_mf["movieId"].map(movie_to_idx)
)

In [23]:
n_factores = 20
learning_rate = 0.01
regularizacion = 0.05
n_epocas = 21

np.random.seed(42)

P = np.random.normal(
    scale=0.1,
    size=(n_usuarios, n_factores)
)

Q = np.random.normal(
    scale=0.1,
    size=(n_peliculas, n_factores)
)

bias_usuarios = np.zeros(n_usuarios)
bias_peliculas = np.zeros(n_peliculas)

media_global = train["rating"].mean()

print("P:", P.shape)
print("Q:", Q.shape)
print("Media global:", round(media_global, 4))

P: (610, 20)
Q: (8983, 20)
Media global: 3.5026


### 4.2. Entrenamiento del modelo

El modelo de factores latentes se entrena mediante descenso de gradiente estocástico. Para cada valoración del conjunto de entrenamiento se calcula la predicción, se obtiene el error cometido y se actualizan los factores latentes y los sesgos correspondientes al usuario y a la película.

Se mantienen los hiperparámetros seleccionados durante el desarrollo previo del modelo, sin realizar nuevos ajustes a partir del conjunto de test.

In [24]:
for epoca in range(n_epocas):

    error_total = 0

    datos_epoca = train_mf.sample(
        frac=1,
        random_state=42 + epoca
    )

    for fila in datos_epoca.itertuples():

        u = fila.user_idx
        i = fila.movie_idx
        rating_real = fila.rating

        prediccion = (
            media_global
            + bias_usuarios[u]
            + bias_peliculas[i]
            + np.dot(P[u], Q[i])
        )

        error = rating_real - prediccion

        error_total += error ** 2

        P_u_anterior = P[u].copy()

        bias_usuarios[u] += learning_rate * (
            error
            - regularizacion * bias_usuarios[u]
        )

        bias_peliculas[i] += learning_rate * (
            error
            - regularizacion * bias_peliculas[i]
        )

        P[u] += learning_rate * (
            error * Q[i]
            - regularizacion * P[u]
        )

        Q[i] += learning_rate * (
            error * P_u_anterior
            - regularizacion * Q[i]
        )

    rmse_train = np.sqrt(
        error_total / len(train_mf)
    )

    print(
        f"Época {epoca + 1}/{n_epocas} "
        f"- RMSE train: {rmse_train:.4f}"
    )

Época 1/21 - RMSE train: 0.9426
Época 2/21 - RMSE train: 0.8892
Época 3/21 - RMSE train: 0.8698
Época 4/21 - RMSE train: 0.8571
Época 5/21 - RMSE train: 0.8476
Época 6/21 - RMSE train: 0.8394
Época 7/21 - RMSE train: 0.8326
Época 8/21 - RMSE train: 0.8260
Época 9/21 - RMSE train: 0.8195
Época 10/21 - RMSE train: 0.8130
Época 11/21 - RMSE train: 0.8061
Época 12/21 - RMSE train: 0.7991
Época 13/21 - RMSE train: 0.7913
Época 14/21 - RMSE train: 0.7830
Época 15/21 - RMSE train: 0.7740
Época 16/21 - RMSE train: 0.7644
Época 17/21 - RMSE train: 0.7543
Época 18/21 - RMSE train: 0.7436
Época 19/21 - RMSE train: 0.7325
Época 20/21 - RMSE train: 0.7215
Época 21/21 - RMSE train: 0.7102


### 4.3. Evaluación sobre el conjunto de test común

Una vez entrenado el modelo, se generan predicciones para las mismas interacciones utilizadas en la evaluación del modelo Deep Learning.

Las predicciones se limitan al intervalo de valoraciones permitido por MovieLens, comprendido entre 0.5 y 5 puntos. Posteriormente se calculan las métricas MAE y RMSE para permitir la comparación entre modelos.

In [25]:
predicciones_mf = []

for fila in test_comun.itertuples():

    u = user_to_idx[fila.userId]
    i = movie_to_idx[fila.movieId]

    prediccion = (
        media_global
        + bias_usuarios[u]
        + bias_peliculas[i]
        + np.dot(P[u], Q[i])
    )

    prediccion = np.clip(
        prediccion,
        0.5,
        5.0
    )

    predicciones_mf.append(prediccion)

predicciones_mf = np.array(predicciones_mf)

print("Número de predicciones:", len(predicciones_mf))
print("Primeras predicciones:", predicciones_mf[:10])

Número de predicciones: 9695
Primeras predicciones: [3.62384871 4.68276061 3.89081898 4.02830204 3.69339073 3.93561665
 3.83087075 4.22991805 3.27921502 3.1426974 ]


In [26]:
y_test_comun = test_comun["rating"].to_numpy()

mae_mf = mean_absolute_error(
    y_test_comun,
    predicciones_mf
)

rmse_mf = np.sqrt(
    mean_squared_error(
        y_test_comun,
        predicciones_mf
    )
)

print("MAE Factores Latentes:", round(mae_mf, 4))
print("RMSE Factores Latentes:", round(rmse_mf, 4))

MAE Factores Latentes: 0.6649
RMSE Factores Latentes: 0.872


#### Resultados

El modelo de factores latentes obtiene un MAE de 0.6649 y un RMSE de 0.8720 sobre el conjunto de test común.

En comparación, el modelo Deep Learning híbrido obtiene valores inferiores en ambas métricas, con un MAE de 0.6462 y un RMSE de 0.8527.

Estos resultados muestran una mejora en la capacidad de predicción de valoraciones del modelo híbrido respecto al modelo basado únicamente en factores latentes. No obstante, la diferencia es moderada, por lo que será necesario complementar este análisis con métricas orientadas a la calidad de las recomendaciones.

## 5. Evaluación del filtrado colaborativo clásico

El tercer modelo evaluado corresponde al sistema de filtrado colaborativo basado en usuarios.

Este método estima la valoración de una película a partir de las valoraciones realizadas por usuarios similares. Para garantizar una comparación homogénea, el modelo se construye utilizando el mismo conjunto de entrenamiento y se evalúa sobre las mismas interacciones del conjunto de test común utilizadas por los modelos anteriores.

### 5.1. Construcción de la matriz usuario-película

A partir del conjunto de entrenamiento se construye una matriz en la que las filas representan usuarios y las columnas películas. Las posiciones conocidas contienen las valoraciones realizadas, mientras que las películas no valoradas permanecen como valores ausentes.

In [27]:
matriz_train = train.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

print("Dimensiones:", matriz_train.shape)

Dimensiones: (610, 8983)


In [28]:
media_usuarios = matriz_train.mean(axis=1)

matriz_train_normalizada = matriz_train.sub(
    media_usuarios,
    axis=0
)

In [29]:
from sklearn.metrics.pairwise import cosine_similarity

matriz_similitud = matriz_train_normalizada.fillna(0)

similitud_train = pd.DataFrame(
    cosine_similarity(matriz_similitud),
    index=matriz_train.index,
    columns=matriz_train.index
)

print("Matriz de similitud:", similitud_train.shape)

Matriz de similitud: (610, 610)


### 5.2. Predicción sobre el conjunto de test común

Para cada interacción del conjunto de test se seleccionan los usuarios más similares al usuario objetivo. La predicción se obtiene a partir de las desviaciones respecto a la media de aquellos vecinos que hayan valorado la película.

Se utilizan 10 vecinos y se exige un mínimo de 3 valoraciones disponibles entre ellos. Cuando no existe evidencia suficiente, el modelo no genera una predicción.

In [30]:
def predecir_colaborativo(
    usuario_id,
    pelicula_id,
    k=10,
    min_vecinos=3
):

    # Si el usuario o la película no existen en train
    if usuario_id not in matriz_train.index:
        return np.nan

    if pelicula_id not in matriz_train.columns:
        return np.nan

    # Similitudes con los demás usuarios
    similitudes_usuario = (
        similitud_train
        .loc[usuario_id]
        .drop(usuario_id)
    )

    # Seleccionamos los k vecinos más similares
    vecinos = (
        similitudes_usuario
        .sort_values(ascending=False)
        .head(k)
    )

    # Valoraciones normalizadas de esos vecinos
    desviaciones = (
        matriz_train_normalizada
        .loc[vecinos.index, pelicula_id]
        .dropna()
    )

    # Exigimos un mínimo de vecinos
    if len(desviaciones) < min_vecinos:
        return np.nan

    # Similitudes de los vecinos que sí valoraron la película
    similitudes_validas = vecinos.loc[desviaciones.index]

    denominador = np.abs(similitudes_validas).sum()

    if denominador == 0:
        return np.nan

    prediccion = (
        media_usuarios.loc[usuario_id]
        + (
            desviaciones * similitudes_validas
        ).sum() / denominador
    )

    return np.clip(prediccion, 0.5, 5.0)

In [31]:
predicciones_cf = []

for fila in test_comun.itertuples():

    prediccion = predecir_colaborativo(
        fila.userId,
        fila.movieId,
        k=10,
        min_vecinos=3
    )

    predicciones_cf.append(prediccion)

predicciones_cf = np.array(predicciones_cf)

In [32]:
mascara_validas_cf = ~np.isnan(predicciones_cf)

n_predicciones_cf = mascara_validas_cf.sum()

cobertura_cf = (
    n_predicciones_cf
    / len(test_comun)
    * 100
)

print("Interacciones solicitadas:", len(test_comun))
print("Predicciones realizadas:", n_predicciones_cf)
print("Predicciones no realizadas:", (~mascara_validas_cf).sum())
print("Cobertura:", round(cobertura_cf, 2), "%")

Interacciones solicitadas: 9695
Predicciones realizadas: 3076
Predicciones no realizadas: 6619
Cobertura: 31.73 %


### 5.3. Métricas de error y cobertura

A diferencia de los modelos de factores latentes y Deep Learning, el filtrado colaborativo clásico no puede generar una predicción para todas las interacciones del conjunto de test.

Con la configuración utilizada, el modelo genera 3.076 predicciones de las 9.695 solicitadas, lo que representa una cobertura del 31,73 %.

Las métricas MAE y RMSE se calculan únicamente sobre las interacciones para las que el modelo ha podido generar una predicción. Por este motivo, estas métricas deben interpretarse conjuntamente con la cobertura del modelo.

In [33]:
y_test_cf = (
    test_comun["rating"]
    .to_numpy()[mascara_validas_cf]
)

predicciones_cf_validas = (
    predicciones_cf[mascara_validas_cf]
)

mae_cf = mean_absolute_error(
    y_test_cf,
    predicciones_cf_validas
)

rmse_cf = np.sqrt(
    mean_squared_error(
        y_test_cf,
        predicciones_cf_validas
    )
)

print(
    "MAE Filtrado Colaborativo:",
    round(mae_cf, 4)
)

print(
    "RMSE Filtrado Colaborativo:",
    round(rmse_cf, 4)
)

print(
    "Cobertura:",
    round(cobertura_cf, 2),
    "%"
)

MAE Filtrado Colaborativo: 0.6325
RMSE Filtrado Colaborativo: 0.8257
Cobertura: 31.73 %


## 6. Comparación directa entre modelos

Las métricas obtenidas inicialmente por el filtrado colaborativo no son directamente comparables con las de los otros modelos, ya que este únicamente puede generar predicciones para una parte del conjunto de test.

Por este motivo, se realiza una segunda evaluación utilizando exclusivamente las interacciones para las que los tres modelos disponen de una predicción. De esta forma, MAE y RMSE se calculan exactamente sobre los mismos casos.

In [34]:
predicciones_mf_comunes = (
    predicciones_mf[mascara_validas_cf]
)

In [35]:
predicciones_dl_comunes = (
    predicciones_dl[mascara_validas_cf]
)

In [36]:
predicciones_dl_comunes = np.clip(
    predicciones_dl_comunes,
    0.5,
    5.0
)

In [37]:
y_comun_tres = (
    test_comun["rating"]
    .to_numpy()[mascara_validas_cf]
)

In [38]:
resultados_comunes = pd.DataFrame({
    "Modelo": [
        "Filtrado colaborativo",
        "Factores latentes",
        "Deep Learning híbrido"
    ],
    "MAE": [
        mean_absolute_error(
            y_comun_tres,
            predicciones_cf_validas
        ),
        mean_absolute_error(
            y_comun_tres,
            predicciones_mf_comunes
        ),
        mean_absolute_error(
            y_comun_tres,
            predicciones_dl_comunes
        )
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(
            y_comun_tres,
            predicciones_cf_validas
        )),
        np.sqrt(mean_squared_error(
            y_comun_tres,
            predicciones_mf_comunes
        )),
        np.sqrt(mean_squared_error(
            y_comun_tres,
            predicciones_dl_comunes
        ))
    ]
})

resultados_comunes.round(4)

,Modelo,MAE,RMSE
0,Filtrado colaborativo,0.6325,0.8257
1,Factores latentes,0.6003,0.7791
2,Deep Learning híbrido,0.5874,0.7681


### 6.1. Resultados sobre las interacciones comunes

La comparación sobre las 3.076 interacciones para las que el filtrado colaborativo puede generar una predicción permite evaluar los tres modelos bajo las mismas condiciones.

El filtrado colaborativo obtiene un MAE de 0.6325 y un RMSE de 0.8257. El modelo de factores latentes mejora estos resultados, alcanzando un MAE de 0.6003 y un RMSE de 0.7791. Finalmente, el modelo Deep Learning híbrido obtiene los menores errores, con un MAE de 0.5874 y un RMSE de 0.7681.

Los resultados muestran una mejora progresiva en la capacidad de predicción desde el filtrado colaborativo clásico hasta el modelo Deep Learning híbrido. Además, el filtrado colaborativo presenta una cobertura del 31.73 %, mientras que los otros modelos pueden generar predicciones para una proporción considerablemente mayor de las interacciones evaluables.

In [39]:
predicciones_dl_limitadas = np.clip(
    predicciones_dl,
    0.5,
    5.0
)

mae_dl_clip = mean_absolute_error(
    y_test,
    predicciones_dl_limitadas
)

rmse_dl_clip = np.sqrt(
    mean_squared_error(
        y_test,
        predicciones_dl_limitadas
    )
)

print("MAE DL con límites:", round(mae_dl_clip, 4))
print("RMSE DL con límites:", round(rmse_dl_clip, 4))

MAE DL con límites: 0.6456
RMSE DL con límites: 0.8524


## 7. Evaluación de recomendaciones Top-N

Las métricas MAE y RMSE permiten evaluar la capacidad de los modelos para predecir las valoraciones de los usuarios. Sin embargo, el objetivo final del sistema es generar listas de películas recomendadas.

Por este motivo, la evaluación se complementa mediante métricas Top-N, que permiten analizar si las películas consideradas relevantes para un usuario aparecen entre las primeras posiciones de las recomendaciones generadas por el modelo.

Para este análisis se utilizarán Precision@K, Recall@K y Hit Rate@K.

### 7.1. Definición de relevancia

Para evaluar las recomendaciones se considera relevante una película cuando el usuario le ha otorgado una valoración igual o superior a 4 puntos en el conjunto de test.

De esta forma, la evaluación Top-N no busca reproducir exactamente la puntuación otorgada por el usuario, sino comprobar si el sistema es capaz de situar entre sus primeras recomendaciones películas que posteriormente han recibido una valoración positiva.

In [40]:
umbral_relevancia = 4.0

test_relevantes = test_comun[
    test_comun["rating"] >= umbral_relevancia
].copy()

print(
    "Interacciones del test común:",
    len(test_comun)
)

print(
    "Interacciones relevantes:",
    len(test_relevantes)
)

print(
    "Porcentaje relevante:",
    round(
        len(test_relevantes)
        / len(test_comun)
        * 100,
        2
    ),
    "%"
)

print(
    "Usuarios con al menos una película relevante:",
    test_relevantes["userId"].nunique()
)

Interacciones del test común: 9695
Interacciones relevantes: 4617
Porcentaje relevante: 47.62 %
Usuarios con al menos una película relevante: 565


### 7.2. Construcción de los conjuntos de candidatos

Para evaluar la capacidad de ranking de los modelos se construye, para cada interacción relevante del conjunto de test, un conjunto de películas candidatas.

Cada conjunto contiene la película relevante, cuya valoración real es igual o superior a 4, junto con 99 películas que el usuario no ha valorado en los conjuntos de entrenamiento, validación o test.

El modelo debe asignar una puntuación a las 100 películas candidatas y situar la película relevante lo más arriba posible en el ranking.

Se fija una semilla aleatoria para garantizar que los tres modelos sean evaluados utilizando exactamente los mismos candidatos.

In [41]:
valoraciones_conocidas = pd.concat(
    [
        train[["userId", "movieId"]],
        validation[["userId", "movieId"]],
        test[["userId", "movieId"]]
    ],
    ignore_index=True
)

peliculas_vistas_usuario = (
    valoraciones_conocidas
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

In [42]:
peliculas_candidatas = np.array(
    sorted(peliculas_train)
)

print(
    "Películas disponibles como candidatas:",
    len(peliculas_candidatas)
)

Películas disponibles como candidatas: 8983


In [43]:
rng = np.random.default_rng(42)

candidatos_topn = []

for fila in test_relevantes.itertuples():

    usuario = fila.userId
    pelicula_relevante = fila.movieId

    vistas = peliculas_vistas_usuario.get(
        usuario,
        set()
    )

    negativas_disponibles = np.array([
        pelicula
        for pelicula in peliculas_candidatas
        if pelicula not in vistas
    ])

    negativas = rng.choice(
        negativas_disponibles,
        size=99,
        replace=False
    )

    candidatos = np.concatenate(
        [
            [pelicula_relevante],
            negativas
        ]
    )

    candidatos_topn.append({
        "userId": usuario,
        "pelicula_relevante": pelicula_relevante,
        "candidatos": candidatos
    })

In [44]:
print(
    "Casos de evaluación:",
    len(candidatos_topn)
)

print(
    "Candidatos del primer caso:",
    len(candidatos_topn[0]["candidatos"])
)

print(
    "Usuario del primer caso:",
    candidatos_topn[0]["userId"]
)

print(
    "Película relevante:",
    candidatos_topn[0]["pelicula_relevante"]
)

Casos de evaluación: 4617
Candidatos del primer caso: 100
Usuario del primer caso: 491
Película relevante: 30825


### 7.3. Evaluación Top-N del modelo de factores latentes

Para cada conjunto de candidatos se calculan las puntuaciones estimadas mediante el modelo de factores latentes.

Las 100 películas se ordenan de mayor a menor puntuación y se comprueba la posición ocupada por la película relevante. A partir de estos rankings se calcula el rendimiento del modelo para K = 10.

In [45]:
def predecir_mf(usuario_id, pelicula_id):

    u = user_to_idx[usuario_id]
    i = movie_to_idx[pelicula_id]

    prediccion = (
        media_global
        + bias_usuarios[u]
        + bias_peliculas[i]
        + np.dot(P[u], Q[i])
    )

    return np.clip(prediccion, 0.5, 5.0)

In [46]:
k = 10

hits_mf = []
rangos_mf = []

for caso in candidatos_topn:

    usuario = caso["userId"]
    pelicula_relevante = caso["pelicula_relevante"]
    candidatos = caso["candidatos"]

    puntuaciones = np.array([
        predecir_mf(usuario, pelicula)
        for pelicula in candidatos
    ])

    orden = np.argsort(-puntuaciones)

    ranking = candidatos[orden]

    posicion = (
        np.where(ranking == pelicula_relevante)[0][0]
        + 1
    )

    rangos_mf.append(posicion)

    hits_mf.append(
        1 if posicion <= k else 0
    )

In [47]:
hit_rate_mf = np.mean(hits_mf)

print(
    "Hit Rate@10 Factores Latentes:",
    round(hit_rate_mf, 4)
)

print(
    "Posición media de la película relevante:",
    round(np.mean(rangos_mf), 2)
)

print(
    "Posición mediana:",
    round(np.median(rangos_mf), 2)
)

Hit Rate@10 Factores Latentes: 0.452
Posición media de la película relevante: 26.58
Posición mediana: 13.0


### 7.4. Evaluación Top-N del modelo Deep Learning híbrido

El modelo Deep Learning híbrido se evalúa utilizando exactamente los mismos conjuntos de candidatos empleados para el modelo de factores latentes.

Para cada interacción se generan las cuatro entradas utilizadas por la red neuronal: usuario, película, géneros de la película y preferencias de género del usuario. Las puntuaciones predichas permiten ordenar las 100 películas candidatas y determinar la posición ocupada por la película relevante.

In [48]:
generos_por_pelicula = (
    movies_generos_test
    .set_index("movieId")[columnas_generos]
)

preferencias_por_usuario = (
    preferencias_usuario
    .set_index("userId")[columnas_preferencias]
)

In [49]:
usuarios_topn = []
peliculas_topn = []
generos_topn = []
preferencias_topn = []

for caso in candidatos_topn:

    usuario = caso["userId"]

    for pelicula in caso["candidatos"]:

        usuarios_topn.append(
            usuario_a_indice[usuario]
        )

        peliculas_topn.append(
            pelicula_a_indice[pelicula]
        )

        generos_topn.append(
            generos_por_pelicula
            .loc[pelicula]
            .to_numpy()
        )

        preferencias_topn.append(
            preferencias_por_usuario
            .loc[usuario]
            .to_numpy()
        )

In [50]:
usuarios_topn = np.array(
    usuarios_topn,
    dtype="int32"
)

peliculas_topn = np.array(
    peliculas_topn,
    dtype="int32"
)

generos_topn = np.array(
    generos_topn,
    dtype="float32"
)

preferencias_topn = np.array(
    preferencias_topn,
    dtype="float32"
)

In [51]:
print("Usuarios:", usuarios_topn.shape)
print("Películas:", peliculas_topn.shape)
print("Géneros:", generos_topn.shape)
print("Preferencias:", preferencias_topn.shape)

Usuarios: (461700,)
Películas: (461700,)
Géneros: (461700, 20)
Preferencias: (461700, 20)


In [52]:
predicciones_topn_dl = modelo_dl.predict(
    [
        usuarios_topn,
        peliculas_topn,
        generos_topn,
        preferencias_topn
    ],
    batch_size=1024,
    verbose=1
).flatten()

predicciones_topn_dl = np.clip(
    predicciones_topn_dl,
    0.5,
    5.0
)

print(
    "Número de predicciones:",
    len(predicciones_topn_dl)
)

451/451 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
Número de predicciones: 461700


In [53]:
predicciones_topn_dl = predicciones_topn_dl.reshape(
    len(candidatos_topn),
    100
)

In [54]:
hits_dl = []
rangos_dl = []

for idx, caso in enumerate(candidatos_topn):

    candidatos = caso["candidatos"]
    pelicula_relevante = caso["pelicula_relevante"]

    puntuaciones = predicciones_topn_dl[idx]

    orden = np.argsort(-puntuaciones)

    ranking = candidatos[orden]

    posicion = (
        np.where(ranking == pelicula_relevante)[0][0]
        + 1
    )

    rangos_dl.append(posicion)

    hits_dl.append(
        1 if posicion <= 10 else 0
    )

In [55]:
hit_rate_dl = np.mean(hits_dl)

print(
    "Hit Rate@10 Deep Learning:",
    round(hit_rate_dl, 4)
)

print(
    "Posición media de la película relevante:",
    round(np.mean(rangos_dl), 2)
)

print(
    "Posición mediana:",
    round(np.median(rangos_dl), 2)
)

Hit Rate@10 Deep Learning: 0.151
Posición media de la película relevante: 32.19
Posición mediana: 27.0


In [56]:
fila = test_comun.iloc[0]

usuario = fila["userId"]
pelicula = fila["movieId"]

u_idx = usuario_a_indice[usuario]
p_idx = pelicula_a_indice[pelicula]

generos = (
    generos_por_pelicula
    .loc[pelicula]
    .to_numpy(dtype="float32")
    .reshape(1, -1)
)

preferencias = (
    preferencias_por_usuario
    .loc[usuario]
    .to_numpy(dtype="float32")
    .reshape(1, -1)
)

prediccion_comprobacion = modelo_dl.predict(
    [
        np.array([u_idx]),
        np.array([p_idx]),
        generos,
        preferencias
    ],
    verbose=0
).flatten()[0]

print("Predicción original:", predicciones_dl[0])
print("Predicción reconstruida:", prediccion_comprobacion)
print("Diferencia:", abs(predicciones_dl[0] - prediccion_comprobacion))

Predicción original: 4.1235876
Predicción reconstruida: 4.123588
Diferencia: 4.7683716e-07


### 7.5. Evaluación de la posición de las recomendaciones

Además del Hit Rate@10, se utiliza NDCG@10 para tener en cuenta la posición ocupada por la película relevante dentro del ranking.

Esta métrica asigna una mayor puntuación cuando la película relevante aparece en las primeras posiciones del Top-10 y reduce progresivamente su contribución conforme aparece en posiciones inferiores. Si la película no se encuentra entre las diez primeras recomendaciones, su contribución es cero.

In [57]:
def ndcg_at_k(rangos, k=10):

    valores = []

    for posicion in rangos:

        if posicion <= k:
            valores.append(
                1 / np.log2(posicion + 1)
            )
        else:
            valores.append(0)

    return np.mean(valores)


ndcg_mf = ndcg_at_k(rangos_mf, 10)
ndcg_dl = ndcg_at_k(rangos_dl, 10)

print(
    "NDCG@10 Factores Latentes:",
    round(ndcg_mf, 4)
)

print(
    "NDCG@10 Deep Learning:",
    round(ndcg_dl, 4)
)

NDCG@10 Factores Latentes: 0.2534
NDCG@10 Deep Learning: 0.0632


### 7.6. Evaluación Top-N del filtrado colaborativo

El filtrado colaborativo se evalúa utilizando los mismos conjuntos de 100 películas candidatas empleados para los modelos anteriores.

A diferencia de los modelos de factores latentes y Deep Learning, este método puede no generar una predicción cuando no existe un número suficiente de vecinos que hayan valorado una determinada película.

Por este motivo, además de las métricas de ranking, se analiza la cobertura del modelo dentro de los conjuntos de candidatos.

In [58]:
predicciones_topn_cf = []

for indice, caso in enumerate(candidatos_topn):

    usuario = caso["userId"]

    puntuaciones = []

    for pelicula in caso["candidatos"]:

        prediccion = predecir_colaborativo(
            usuario,
            pelicula,
            k=10,
            min_vecinos=3
        )

        puntuaciones.append(prediccion)

    predicciones_topn_cf.append(
        np.array(puntuaciones)
    )

    if (indice + 1) % 500 == 0:
        print(
            f"Procesados {indice + 1} "
            f"de {len(candidatos_topn)} casos"
        )

Procesados 500 de 4617 casos
Procesados 1000 de 4617 casos
Procesados 1500 de 4617 casos
Procesados 2000 de 4617 casos
Procesados 2500 de 4617 casos
Procesados 3000 de 4617 casos
Procesados 3500 de 4617 casos
Procesados 4000 de 4617 casos
Procesados 4500 de 4617 casos


In [59]:
total_candidatos_cf = 0
total_predicciones_cf = 0

relevantes_predecibles_cf = 0

for caso, puntuaciones in zip(
    candidatos_topn,
    predicciones_topn_cf
):

    total_candidatos_cf += len(puntuaciones)

    validas = ~np.isnan(puntuaciones)

    total_predicciones_cf += validas.sum()

    pelicula_relevante = caso["pelicula_relevante"]

    indice_relevante = np.where(
        caso["candidatos"] == pelicula_relevante
    )[0][0]

    if validas[indice_relevante]:
        relevantes_predecibles_cf += 1


cobertura_candidatos_cf = (
    total_predicciones_cf
    / total_candidatos_cf
)

cobertura_relevantes_cf = (
    relevantes_predecibles_cf
    / len(candidatos_topn)
)

print(
    "Cobertura sobre candidatos:",
    round(cobertura_candidatos_cf * 100, 2),
    "%"
)

print(
    "Películas relevantes predecibles:",
    relevantes_predecibles_cf,
    "de",
    len(candidatos_topn)
)

print(
    "Cobertura sobre películas relevantes:",
    round(cobertura_relevantes_cf * 100, 2),
    "%"
)

Cobertura sobre candidatos: 2.3 %
Películas relevantes predecibles: 1592 de 4617
Cobertura sobre películas relevantes: 34.48 %


#### Resultados Top-N del filtrado colaborativo

Las películas para las que el modelo no puede generar una predicción se excluyen del ranking. Esta limitación se mantiene como parte de la evaluación, ya que representa una característica real del funcionamiento del filtrado colaborativo utilizado.

Una interacción cuya película relevante no puede ser predicha se considera un fallo en la evaluación global, puesto que el sistema sería incapaz de recomendar dicha película.

In [60]:
hits_cf = []
rangos_cf = []

for caso, puntuaciones in zip(
    candidatos_topn,
    predicciones_topn_cf
):

    candidatos = caso["candidatos"]
    pelicula_relevante = caso["pelicula_relevante"]

    validas = ~np.isnan(puntuaciones)

    candidatos_validos = candidatos[validas]
    puntuaciones_validas = puntuaciones[validas]

    # Si la película relevante no puede predecirse,
    # consideramos que no existe hit
    if pelicula_relevante not in candidatos_validos:

        hits_cf.append(0)
        rangos_cf.append(np.nan)

        continue

    orden = np.argsort(
        -puntuaciones_validas
    )

    ranking = candidatos_validos[orden]

    posicion = (
        np.where(
            ranking == pelicula_relevante
        )[0][0]
        + 1
    )

    rangos_cf.append(posicion)

    hits_cf.append(
        1 if posicion <= 10 else 0
    )

In [61]:
hit_rate_cf = np.mean(hits_cf)

print(
    "Hit Rate@10 Filtrado Colaborativo:",
    round(hit_rate_cf, 4)
)

Hit Rate@10 Filtrado Colaborativo: 0.3439


In [62]:
ndcg_cf_valores = []

for posicion in rangos_cf:

    if (
        not np.isnan(posicion)
        and posicion <= 10
    ):

        ndcg_cf_valores.append(
            1 / np.log2(posicion + 1)
        )

    else:

        ndcg_cf_valores.append(0)


ndcg_cf = np.mean(
    ndcg_cf_valores
)

print(
    "NDCG@10 Filtrado Colaborativo:",
    round(ndcg_cf, 4)
)

NDCG@10 Filtrado Colaborativo: 0.2826


In [63]:
rangos_cf_validos = np.array([
    posicion
    for posicion in rangos_cf
    if not np.isnan(posicion)
])

hit_rate_cf_condicional = np.mean(
    rangos_cf_validos <= 10
)

print(
    "Hit Rate@10 sobre relevantes predecibles:",
    round(hit_rate_cf_condicional, 4)
)

print(
    "Posición media cuando puede predecir:",
    round(np.mean(rangos_cf_validos), 2)
)

print(
    "Posición mediana cuando puede predecir:",
    round(np.median(rangos_cf_validos), 2)
)

Hit Rate@10 sobre relevantes predecibles: 0.9975
Posición media cuando puede predecir: 1.87
Posición mediana cuando puede predecir: 1.0


In [64]:
num_validas_por_caso = np.array([
    np.sum(~np.isnan(puntuaciones))
    for puntuaciones in predicciones_topn_cf
])

print(
    "Predicciones válidas medias por 100 candidatos:",
    round(np.mean(num_validas_por_caso), 2)
)

print(
    "Mediana:",
    np.median(num_validas_por_caso)
)

print(
    "Casos con 10 o menos predicciones:",
    np.sum(num_validas_por_caso <= 10),
    "de",
    len(num_validas_por_caso)
)

print(
    "Porcentaje:",
    round(
        np.mean(num_validas_por_caso <= 10) * 100,
        2
    ),
    "%"
)

Predicciones válidas medias por 100 candidatos: 2.3
Mediana: 1.0
Casos con 10 o menos predicciones: 4506 de 4617
Porcentaje: 97.6 %


In [65]:
hit_rate_cf = np.mean(hits_cf)

print(
    "Hit Rate@10 Filtrado Colaborativo:",
    round(hit_rate_cf, 4)
)

Hit Rate@10 Filtrado Colaborativo: 0.3439


In [66]:
num_validas_por_caso = np.array([
    np.sum(~np.isnan(puntuaciones))
    for puntuaciones in predicciones_topn_cf
])

print(
    "Predicciones válidas medias por 100 candidatos:",
    round(np.mean(num_validas_por_caso), 2)
)

print(
    "Mediana:",
    np.median(num_validas_por_caso)
)

print(
    "Casos con 10 o menos predicciones:",
    np.sum(num_validas_por_caso <= 10),
    "de",
    len(num_validas_por_caso)
)

print(
    "Porcentaje con 10 o menos:",
    round(
        np.mean(num_validas_por_caso <= 10) * 100,
        2
    ),
    "%"
)

print(
    "Máximo de predicciones en un caso:",
    np.max(num_validas_por_caso)
)

Predicciones válidas medias por 100 candidatos: 2.3
Mediana: 1.0
Casos con 10 o menos predicciones: 4506 de 4617
Porcentaje con 10 o menos: 97.6 %
Máximo de predicciones en un caso: 19
